# ECE 3715 Mini-Project 3

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from scipy import integrate, stats
from scipy.stats import multivariate_normal

from mp3lib import (
    cholesky_samples,
    eigen_samples,
    gaussian_pdf,
    is_valid_covariance,
    lmmse_weights,
    sample_mean_cov,
    whitening_matrix,
)

# The handout prints this value as its example. Replace it with the
# one-time team draw if the instructor requires unique team seeds.
SEED = 1734829102
rng = np.random.default_rng(SEED)
STEP_RNGS = rng.spawn(8)
rng_step1, rng_step2, rng_step3, rng_step4, rng_step5, rng_step6, rng_step7, rng_step8 = STEP_RNGS
BONUS_RNG_A, BONUS_RNG_B = rng.spawn(2)

np.set_printoptions(precision=6, suppress=True, linewidth=120)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 8,
    }
)
START_TIME = time.perf_counter()
print(f"SEED = {SEED}")
print(f"Spawned {len(STEP_RNGS)} independent step generators.")

In [ ]:
def batch_stat_se(data, statistic, n_batches=100):
    # Estimate a statistic and its Monte Carlo standard error with batch means.
    data = np.asarray(data)
    n = data.shape[0]
    n_batches = min(n_batches, n)
    edges = np.linspace(0, n, n_batches + 1, dtype=int)
    batch_values = []
    for start, stop in zip(edges[:-1], edges[1:]):
        if stop > start:
            batch_values.append(np.asarray(statistic(data[start:stop])))
    batch_values = np.stack(batch_values)
    estimate = np.asarray(statistic(data))
    standard_error = np.std(batch_values, axis=0, ddof=1) / np.sqrt(len(batch_values))
    return estimate, standard_error


def covariance_se(data, n_batches=100):
    return batch_stat_se(data, lambda block: sample_mean_cov(block)[1], n_batches)


def mahalanobis_squared(X, mu, Sigma):
    X = np.asarray(X, dtype=float)
    delta = X - np.asarray(mu, dtype=float)
    solved = np.linalg.solve(Sigma, delta.T).T
    return np.einsum("ij,ij->i", delta, solved, optimize=True)


def binned_moments(x, y, edges):
    x = np.asarray(x)
    y = np.asarray(y)
    bin_index = np.digitize(x, edges) - 1
    n_bins = len(edges) - 1
    valid = (bin_index >= 0) & (bin_index < n_bins)
    bin_index = bin_index[valid]
    y = y[valid]
    counts = np.bincount(bin_index, minlength=n_bins).astype(float)
    sums = np.bincount(bin_index, weights=y, minlength=n_bins)
    sums_sq = np.bincount(bin_index, weights=y * y, minlength=n_bins)
    means = np.full(n_bins, np.nan)
    variances = np.full(n_bins, np.nan)
    standard_errors = np.full(n_bins, np.nan)
    enough = counts >= 2
    means[enough] = sums[enough] / counts[enough]
    variances[enough] = (
        sums_sq[enough] - sums[enough] ** 2 / counts[enough]
    ) / (counts[enough] - 1)
    standard_errors[enough] = np.sqrt(variances[enough] / counts[enough])
    centers = (edges[:-1] + edges[1:]) / 2.0
    return centers, counts, means, variances, standard_errors

## Step 1 — Two variables at once

For the unit-square density

$$f_{X,Y}(x,y)=x+y,\qquad 0\le x\le 1,\quad 0\le y\le 1,$$

normalization is

$$\int_0^1\int_0^1(x+y)\,dy\,dx
=\int_0^1\left(x+\frac12\right)dx=1.$$

The marginals are

$$f_X(x)=\int_0^1(x+y)\,dy=x+\frac12,\qquad
f_Y(y)=y+\frac12.$$

Therefore

$$f_{Y\mid X}(y\mid x)=\frac{x+y}{x+\frac12},\qquad
f_{Y\mid X}(y\mid 0.3)=\frac{0.3+y}{0.8}.$$

The moments needed below follow from the same joint integral:

$$E[X]=E[Y]=\frac{7}{12},\qquad E[XY]=\frac13,$$
$$E[X^2]=E[Y^2]=\frac{5}{12},\qquad
\operatorname{Var}(X)=\operatorname{Var}(Y)=\frac{11}{144},$$
$$\operatorname{Cov}(X,Y)=\frac13-\left(\frac{7}{12}\right)^2
=-\frac1{144},\qquad \rho_{X,Y}=-\frac1{11}.$$

In [ ]:
print("STEP 1: two variables at once")

def joint_density_xy(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    inside = (0 <= x) & (x <= 1) & (0 <= y) & (y <= 1)
    return np.where(inside, x + y, 0.0)


def unit_square_integral(integrand):
    return integrate.dblquad(
        lambda y, x: integrand(x, y),
        0.0,
        1.0,
        lambda _: 0.0,
        lambda _: 1.0,
    )


joint_integral, joint_quad_error = unit_square_integral(lambda x, y: x + y)
x_grid = np.linspace(0.0, 1.0, 401)
y_grid = np.linspace(0.0, 1.0, 401)
X_GRID, Y_GRID = np.meshgrid(x_grid, y_grid, indexing="ij")
joint_grid = joint_density_xy(X_GRID, Y_GRID)
marginal_x_numeric = np.trapezoid(joint_grid, y_grid, axis=1)
marginal_y_numeric = np.trapezoid(joint_grid, x_grid, axis=0)
marginal_x_analytic = x_grid + 0.5
marginal_y_analytic = y_grid + 0.5
marginal_x_max_error = np.max(np.abs(marginal_x_numeric - marginal_x_analytic))
marginal_y_max_error = np.max(np.abs(marginal_y_numeric - marginal_y_analytic))

x_conditional = 0.3
conditional_grid = (x_conditional + y_grid) / (x_conditional + 0.5)
conditional_integral = np.trapezoid(conditional_grid, y_grid)

moment_integrands = {
    "E[X]": lambda x, y: x * (x + y),
    "E[Y]": lambda x, y: y * (x + y),
    "E[XY]": lambda x, y: x * y * (x + y),
    "E[X^2]": lambda x, y: x * x * (x + y),
    "E[Y^2]": lambda x, y: y * y * (x + y),
}
quadrature_moments = {
    name: unit_square_integral(integrand)[0]
    for name, integrand in moment_integrands.items()
}
analytic_moments = {
    "E[X]": 7.0 / 12.0,
    "E[Y]": 7.0 / 12.0,
    "E[XY]": 1.0 / 3.0,
    "E[X^2]": 5.0 / 12.0,
    "E[Y^2]": 5.0 / 12.0,
}
var_x = analytic_moments["E[X^2]"] - analytic_moments["E[X]"] ** 2
var_y = analytic_moments["E[Y^2]"] - analytic_moments["E[Y]"] ** 2
cov_xy = analytic_moments["E[XY]"] - analytic_moments["E[X]"] * analytic_moments["E[Y]"]
rho_xy = cov_xy / np.sqrt(var_x * var_y)
var_x_quad = quadrature_moments["E[X^2]"] - quadrature_moments["E[X]"] ** 2
var_y_quad = quadrature_moments["E[Y^2]"] - quadrature_moments["E[Y]"] ** 2
cov_xy_quad = quadrature_moments["E[XY]"] - quadrature_moments["E[X]"] * quadrature_moments["E[Y]"]
rho_xy_quad = cov_xy_quad / np.sqrt(var_x_quad * var_y_quad)

def rejection_sample_xy(rng_local, n_samples, bound=2.0, batch_size=500_000):
    samples = np.empty((n_samples, 2))
    accepted = 0
    proposals = 0
    while accepted < n_samples:
        candidates = rng_local.random((batch_size, 2))
        uniforms = rng_local.random(batch_size)
        keep = uniforms * bound <= np.sum(candidates, axis=1)
        n_keep = min(int(np.sum(keep)), n_samples - accepted)
        if n_keep:
            samples[accepted : accepted + n_keep] = candidates[keep][:n_keep]
            accepted += n_keep
        proposals += batch_size
    return samples, accepted / proposals


samples_step1, acceptance_rate = rejection_sample_xy(rng_step1, 10**6, bound=2.0)
sample_corr_step1 = np.corrcoef(samples_step1, rowvar=False)[0, 1]
sample_stat_specs = [
    ("E[X]", lambda block: np.mean(block[:, 0]), analytic_moments["E[X]"]),
    ("E[Y]", lambda block: np.mean(block[:, 1]), analytic_moments["E[Y]"]),
    ("E[XY]", lambda block: np.mean(block[:, 0] * block[:, 1]), analytic_moments["E[XY]"]),
    ("Var(X)", lambda block: np.var(block[:, 0], ddof=1), var_x),
    ("Cov(X,Y)", lambda block: np.cov(block[:, 0], block[:, 1], ddof=1)[0, 1], cov_xy),
    ("rho", lambda block: np.corrcoef(block[:, 0], block[:, 1])[0, 1], rho_xy),
]

print(f"Joint-density quadrature = {joint_integral:.12f}; quadrature error estimate = {joint_quad_error:.3e}")
print(f"Conditional integral at x = 0.3 = {conditional_integral:.12f}")
print(f"Marginal max errors: f_X = {marginal_x_max_error:.3e}, f_Y = {marginal_y_max_error:.3e}")
print(f"Analytic Var(X) = {var_x:.12f}; Cov(X,Y) = {cov_xy:.12f}; rho = {rho_xy:.12f}")
print(f"Monte Carlo correlation from accepted samples = {sample_corr_step1:.12f}")
print(f"Quadrature Var(X) = {var_x_quad:.12f}; Cov(X,Y) = {cov_xy_quad:.12f}; rho = {rho_xy_quad:.12f}")
print(f"Rejection bound M = 2.0; acceptance rate = {acceptance_rate:.6f}")
for name, statistic, target in sample_stat_specs:
    estimate, standard_error = batch_stat_se(samples_step1, statistic, n_batches=100)
    print(
        f"{name}: sample = {float(estimate):.9f}, theory = {target:.9f}, "
        f"difference = {float(estimate - target):+.3e}, SE = {float(standard_error):.3e}"
    )

figure1, axes1 = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
contour = axes1[0].contourf(X_GRID, Y_GRID, joint_grid, levels=20, cmap="viridis")
figure1.colorbar(contour, ax=axes1[0], label="f_XY(x, y)")
axes1[0].set_title("Figure 1a: joint density")
axes1[0].set_xlabel("x")
axes1[0].set_ylabel("y")
axes1[1].plot(x_grid, marginal_x_analytic, label="analytic f_X(x)")
axes1[1].plot(x_grid, marginal_x_numeric, "--", label="grid f_X(x)")
axes1[1].plot(y_grid, marginal_y_analytic, label="analytic f_Y(y)")
axes1[1].plot(y_grid, marginal_y_numeric, "--", label="grid f_Y(y)")
axes1[1].set_title("Figure 1b: marginals")
axes1[1].set_xlabel("argument")
axes1[1].set_ylabel("density")
axes1[1].legend()
axes1[2].plot(y_grid, conditional_grid, label="f(y | x = 0.3)")
axes1[2].axhline(0, color="black", linewidth=0.6)
axes1[2].set_title("Figure 1c: conditional density")
axes1[2].set_xlabel("y")
axes1[2].set_ylabel("density")
axes1[2].legend()
plt.show()

Figure 1 interpretation. The heatmap increases toward the upper and
right edges, so the joint distribution gives more mass to large values
of either coordinate. The two marginal curves agree with numerical
integration, and the conditional curve remains normalized after
fixing the value of x.

## Step 2 — Uncorrelated does not mean independent

Let X be uniform on (-1, 1) and let Y = X^2. Since the distribution of
X is symmetric,

$$E[X]=0,\qquad E[X^3]=0,\qquad E[X^2]=\frac13,$$

and therefore

$$\operatorname{Cov}(X,Y)
=E[X^3]-E[X]E[X^2]=0.$$

Nevertheless, Y is completely determined by X, so the conditional
variance is zero. Correlation measures only linear dependence; it is
blind to this symmetric quadratic relationship. For a jointly
Gaussian vector, the covariance matrix fully determines the joint
density, which is why zero cross-covariance then implies independence.

In [ ]:
print("STEP 2: uncorrelated does not mean independent")
n_step2 = 10**7
x_step2 = rng_step2.uniform(-1.0, 1.0, size=n_step2)
y_step2 = x_step2 * x_step2
sample_corr_step2 = np.corrcoef(x_step2, y_step2)[0, 1]
correlation_se_step2 = 1.0 / np.sqrt(n_step2 - 1)

edges_step2 = np.linspace(-1.0, 1.0, 201)
centers2, counts2, means2, variances2, variances_se2 = binned_moments(
    x_step2, y_step2, edges_step2
)
valid2 = np.isfinite(variances2)
max_binned_variance2 = np.nanmax(np.abs(variances2))

n_gaussian_pair = 200_000
gaussian_pair = rng_step2.multivariate_normal(
    np.zeros(2), np.eye(2), size=n_gaussian_pair
)
gaussian_pair_correlation = np.corrcoef(gaussian_pair, rowvar=False)
histogram_edges = np.linspace(-4.0, 4.0, 41)
empirical_joint, _, _ = np.histogram2d(
    gaussian_pair[:, 0],
    gaussian_pair[:, 1],
    bins=(histogram_edges, histogram_edges),
    density=True,
)
empirical_x, _ = np.histogram(gaussian_pair[:, 0], bins=histogram_edges, density=True)
empirical_y, _ = np.histogram(gaussian_pair[:, 1], bins=histogram_edges, density=True)
product_of_marginals = empirical_x[:, None] * empirical_y[None, :]
bin_area = (histogram_edges[1] - histogram_edges[0]) ** 2
gaussian_joint_rmse = np.sqrt(np.mean((empirical_joint - product_of_marginals) ** 2))
gaussian_joint_l1 = np.sum(np.abs(empirical_joint - product_of_marginals)) * bin_area

print(f"N = {n_step2}; sample correlation = {sample_corr_step2:.9f}; null SE = {correlation_se_step2:.3e}")
print(f"Independent Gaussian sample correlation matrix =\n{gaussian_pair_correlation}")
print(f"Max binned Var(Y|X) = {max_binned_variance2:.3e}")
print(
    f"Gaussian rho = 0 comparison: histogram RMSE = {gaussian_joint_rmse:.3e}, "
    f"approximate L1 difference = {gaussian_joint_l1:.3e}"
)

figure2, axes2 = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
scatter_slice = slice(0, min(50_000, n_step2))
axes2[0].scatter(
    x_step2[scatter_slice],
    y_step2[scatter_slice],
    s=2,
    alpha=0.22,
    label="samples",
)
axes2[0].set_title(f"Figure 2a: Y = X²; sample r = {sample_corr_step2:.3e}")
axes2[0].set_xlabel("X")
axes2[0].set_ylabel("Y")
axes2[0].legend()
axes2[1].plot(centers2[valid2], variances2[valid2], "o-", label="binned variance")
axes2[1].axhline(0.0, color="black", linestyle="--", label="theory: 0")
axes2[1].set_title("Figure 2b: Var(Y | X = x)")
axes2[1].set_xlabel("x-bin center")
axes2[1].set_ylabel("conditional variance")
axes2[1].legend()
plt.show()

figure2b, axes2b = plt.subplots(1, 3, figsize=(14, 4.0), constrained_layout=True)
image0 = axes2b[0].imshow(
    empirical_joint.T,
    origin="lower",
    extent=[-4, 4, -4, 4],
    aspect="equal",
    cmap="magma",
)
figure2b.colorbar(image0, ax=axes2b[0], label="density")
axes2b[0].set_title("Figure 2c: empirical joint")
image1 = axes2b[1].imshow(
    product_of_marginals.T,
    origin="lower",
    extent=[-4, 4, -4, 4],
    aspect="equal",
    cmap="magma",
)
figure2b.colorbar(image1, ax=axes2b[1], label="density")
axes2b[1].set_title("Figure 2d: product of marginals")
image2 = axes2b[2].imshow(
    (empirical_joint - product_of_marginals).T,
    origin="lower",
    extent=[-4, 4, -4, 4],
    aspect="equal",
    cmap="coolwarm",
)
figure2b.colorbar(image2, ax=axes2b[2], label="difference")
axes2b[2].set_title("Figure 2e: joint minus product")
for axis in axes2b:
    axis.set_xlabel("x")
    axis.set_ylabel("y")
plt.show()

Figure 2 interpretation. The quadratic scatter is a deterministic
curve even though its linear correlation is near zero, and each
conditional-variance bin is numerically zero. In the Gaussian
comparison, the empirical joint density and the product of empirical
marginals agree up to finite-sample histogram noise.

## Step 3 — The covariance matrix and estimation error

The target covariance is

$$\Sigma=
\begin{bmatrix}
4&2&1&0\\
2&5&2&1\\
1&2&3&1\\
0&1&1&2
\end{bmatrix}.$$

For observations X_i, the unbiased sample covariance is

$$\widehat{\mu}=\frac1N\sum_{i=1}^N X_i,\qquad
\widehat{\Sigma}=\frac1{N-1}\sum_{i=1}^N
(X_i-\widehat{\mu})(X_i-\widehat{\mu})^T.$$

Averaging repeated estimates makes the expected 1/sqrt(N) scaling
visible; a single covariance draw contains enough random variation to
obscure a slope fit.

In [ ]:
print("STEP 3: covariance estimation")
Sigma4 = np.array(
    [
        [4.0, 2.0, 1.0, 0.0],
        [2.0, 5.0, 2.0, 1.0],
        [1.0, 2.0, 3.0, 1.0],
        [0.0, 1.0, 1.0, 2.0],
    ]
)
eigenvalues4 = np.linalg.eigvalsh(Sigma4)
determinant4 = np.linalg.det(Sigma4)
nonsymmetric_test = Sigma4.copy()
nonsymmetric_test[0, 1] += 0.1
negative_eigenvalue_test = np.diag([1.0, 1.0, 1.0, -0.1])
standard_deviations4 = np.sqrt(np.diag(Sigma4))
correlation4 = Sigma4 / np.outer(standard_deviations4, standard_deviations4)
print(f"Sigma valid = {is_valid_covariance(Sigma4)}")
print(f"Sigma eigenvalues = {eigenvalues4}")
print(f"det(Sigma) = {determinant4:.9f}")
print(f"Target correlation matrix =\n{correlation4}")
print(f"Non-symmetric test valid = {is_valid_covariance(nonsymmetric_test)}")
print(
    f"Symmetric negative-eigenvalue test valid = "
    f"{is_valid_covariance(negative_eigenvalue_test)}"
)

sample_sizes3 = np.array([10**2, 10**3, 10**4, 10**5, 10**6])
single_draw_errors3 = []
for n_samples in sample_sizes3:
    draw = rng_step3.multivariate_normal(np.zeros(4), Sigma4, size=int(n_samples))
    _, estimated_covariance = sample_mean_cov(draw)
    single_draw_errors3.append(np.linalg.norm(estimated_covariance - Sigma4, ord="fro"))
single_draw_errors3 = np.asarray(single_draw_errors3)
estimated_standard_deviations3 = np.sqrt(np.diag(estimated_covariance))
estimated_correlation3 = estimated_covariance / np.outer(estimated_standard_deviations3, estimated_standard_deviations3)
print(f"Correlation matrix for the final N={sample_sizes3[-1]} estimate =\n{estimated_correlation3}")
print("Single-draw Frobenius errors:")
for n_samples, error in zip(sample_sizes3, single_draw_errors3):
    print(f"  N = {n_samples:>7d}: error = {error:.6f}")

repetition_counts3 = [200, 200, 100, 30, 10]
repeated_errors3 = []
for n_samples, n_repetitions in zip(sample_sizes3, repetition_counts3):
    errors = np.empty(n_repetitions)
    for repetition in range(n_repetitions):
        draw = rng_step3.multivariate_normal(np.zeros(4), Sigma4, size=int(n_samples))
        _, estimated_covariance = sample_mean_cov(draw)
        errors[repetition] = np.linalg.norm(estimated_covariance - Sigma4, ord="fro")
    repeated_errors3.append(errors)
average_errors3 = np.array([np.mean(errors) for errors in repeated_errors3])
average_error_se3 = np.array(
    [np.std(errors, ddof=1) / np.sqrt(len(errors)) for errors in repeated_errors3]
)
slope3, intercept3 = np.polyfit(np.log10(sample_sizes3), np.log10(average_errors3), 1)
sqrt_n_product3 = np.sqrt(sample_sizes3) * average_errors3
free_parameters_d4 = 4 * 5 // 2
free_parameters_d100 = 100 * 101 // 2
estimated_n_d100 = int(round(10**4 * free_parameters_d100 / free_parameters_d4))
print(f"Averaged error slope = {slope3:.6f}")
print("Averaged errors and standard errors:")
for n_samples, average, standard_error, product in zip(
    sample_sizes3, average_errors3, average_error_se3, sqrt_n_product3
):
    print(
        f"  N = {n_samples:>7d}: mean error = {average:.6f}, "
        f"SE = {standard_error:.3e}, sqrt(N)*error = {product:.6f}"
    )
print(
    f"Free covariance parameters: d=4 -> {free_parameters_d4}; "
    f"d=100 -> {free_parameters_d100}"
)
print(
    f"Parameter-count scaling estimate for d=100 at the d=4, N=10^4 "
    f"relative accuracy: N about {estimated_n_d100:,}"
)

figure3, axes3 = plt.subplots(1, 2, figsize=(12, 4.3), constrained_layout=True)
axes3[0].errorbar(
    sample_sizes3,
    average_errors3,
    yerr=average_error_se3,
    fmt="o-",
    capsize=3,
    label="averaged Frobenius error",
)
fitted_curve3 = 10**intercept3 * sample_sizes3**slope3
axes3[0].loglog(
    sample_sizes3,
    fitted_curve3,
    "--",
    label=f"fit slope = {slope3:.3f}",
)
axes3[0].set_title("Figure 3a: covariance error")
axes3[0].set_xlabel("N")
axes3[0].set_ylabel("Frobenius error")
axes3[0].legend()
axes3[1].semilogx(
    sample_sizes3,
    sqrt_n_product3,
    "o-",
    label="sqrt(N) times averaged error",
)
axes3[1].axhline(
    np.mean(sqrt_n_product3),
    color="black",
    linestyle="--",
    label="mean product",
)
axes3[1].set_title("Figure 3b: scaled error")
axes3[1].set_xlabel("N")
axes3[1].set_ylabel("sqrt(N) * error")
axes3[1].legend()
plt.show()

Figure 3 interpretation. The averaged covariance error decreases
approximately as a power law with exponent near one-half in magnitude.
Multiplying by the square root of the sample size makes the curve
comparatively flat, while the single-draw values are too noisy to
support a reliable slope estimate. The number of free covariance
parameters grows quadratically with dimension, which makes a
100-element array difficult to estimate without more data, structure,
or regularization.

## Step 4 — Linear transformations of random vectors

Let

$$Y=AX+b,\qquad E[X]=\mu_X,\qquad \operatorname{Cov}(X)=\Sigma_X.$$

Then

$$E[Y]=E[AX+b]=AE[X]+b=A\mu_X+b.$$

Centering the output gives

$$Y-E[Y]=A(X-\mu_X),$$
$$\operatorname{Cov}(Y)
=E[(Y-E[Y])(Y-E[Y])^T]
=A\Sigma_XA^T.$$

In [ ]:
print("STEP 4: linear transformations")
mu_x = np.array([1.0, -2.0])
Sigma_x = np.array([[4.0, 2.0], [2.0, 4.0]])
A_full = np.array([[1.0, 2.0], [0.0, 1.0], [1.0, -1.0]])
A_rank = np.array([[1.0, 1.0], [2.0, 2.0]])
mu_y_full = A_full @ mu_x
Sigma_y_full = A_full @ Sigma_x @ A_full.T
mu_y_rank = A_rank @ mu_x
Sigma_y_rank = A_rank @ Sigma_x @ A_rank.T
print(f"Analytic mu_y for full A = {mu_y_full}")
print(f"Analytic Sigma_y for full A =\n{Sigma_y_full}")
print(f"rank(Sigma_y) for full A = {np.linalg.matrix_rank(Sigma_y_full)}")
print(f"Analytic mu_y for rank-deficient A = {mu_y_rank}")
print(f"Analytic Sigma_y for rank-deficient A =\n{Sigma_y_rank}")
print(f"rank(Sigma_y) for rank-deficient A = {np.linalg.matrix_rank(Sigma_y_rank)}")

n_step4 = 10**6
x_samples4 = rng_step4.multivariate_normal(mu_x, Sigma_x, size=n_step4)
y_samples4 = x_samples4 @ A_full.T
sample_mu_y_full, sample_Sigma_y_full = sample_mean_cov(y_samples4)
mean_se4 = np.std(y_samples4, axis=0, ddof=1) / np.sqrt(n_step4)
_, covariance_se4 = covariance_se(y_samples4, n_batches=100)
correlation_x4 = Sigma_x / np.outer(np.sqrt(np.diag(Sigma_x)), np.sqrt(np.diag(Sigma_x)))
correlation_y_full4 = Sigma_y_full / np.outer(np.sqrt(np.diag(Sigma_y_full)), np.sqrt(np.diag(Sigma_y_full)))
correlation_y_sample4 = sample_Sigma_y_full / np.outer(np.sqrt(np.diag(sample_Sigma_y_full)), np.sqrt(np.diag(sample_Sigma_y_full)))
print(f"Sample mu_y = {sample_mu_y_full}")
print(f"Mean standard errors = {mean_se4}")
print(f"Mean difference = {sample_mu_y_full - mu_y_full}")
print(f"Sample Sigma_y =\n{sample_Sigma_y_full}")
print(f"Covariance standard errors by entry =\n{covariance_se4}")
print(f"Covariance Frobenius difference = {np.linalg.norm(sample_Sigma_y_full - Sigma_y_full):.3e}")
print(f"Correlation of input X =\n{correlation_x4}")
print(f"Analytic correlation of full output Y =\n{correlation_y_full4}")
print(f"Sample correlation of full output Y =\n{correlation_y_sample4}")

y_rank_samples4 = x_samples4 @ A_rank.T
singular_determinant = np.linalg.det(Sigma_y_rank)
print(f"det(Sigma_y) for rank-deficient A = {singular_determinant:.3e}")
try:
    gaussian_pdf(np.zeros(2), mu_y_rank, Sigma_y_rank)
except ValueError as error:
    print(f"Singular Gaussian-density check: {error}")

figure4, axis4 = plt.subplots(figsize=(6, 5))
rank_plot_slice = slice(0, 50_000)
axis4.scatter(
    y_rank_samples4[rank_plot_slice, 0],
    y_rank_samples4[rank_plot_slice, 1],
    s=3,
    alpha=0.2,
    label="rank-deficient output samples",
)
line_parameter = np.linspace(-8.0, 4.0, 100)
axis4.plot(
    line_parameter,
    2.0 * line_parameter,
    color="black",
    linewidth=2,
    label="collapse line: y_2 = 2 y_1",
)
axis4.set_aspect("equal", adjustable="box")
axis4.set_title("Figure 4: output of rank-deficient A")
axis4.set_xlabel("y_1")
axis4.set_ylabel("y_2")
axis4.legend()
plt.show()

Figure 4 interpretation. The rank-deficient transformation maps every
input vector onto a one-dimensional line, so the output covariance has
rank one even though it is a two-by-two matrix. A singular covariance
has zero determinant, so the ordinary full-dimensional Gaussian
density formula cannot be evaluated.

## Step 5 — The multivariate Gaussian

For a nonsingular d-dimensional covariance matrix,

$$f_X(x)=\frac{1}{(2\pi)^{d/2}\det(\Sigma)^{1/2}}
\exp\left[-\frac12(x-\mu)^T\Sigma^{-1}(x-\mu)\right].$$

The determinant controls the normalizing volume, while the quadratic
form is the squared Mahalanobis distance. The implementation uses
slogdet and solve, which evaluate the same expression without
explicitly constructing an inverse matrix.

In [ ]:
print("STEP 5: multivariate Gaussian")
mu2 = np.zeros(2)
Sigma2 = np.array([[4.0, 2.0], [2.0, 4.0]])
test_points5 = np.array(
    [
        [-1.0, 0.0],
        [0.0, 1.0],
        [2.0, -1.0],
        [3.0, 2.0],
        [-2.0, -2.0],
    ]
)
ours5 = gaussian_pdf(test_points5, mu2, Sigma2)
scipy5 = multivariate_normal(mean=mu2, cov=Sigma2).pdf(test_points5)
relative_difference5 = np.abs(ours5 - scipy5) / np.maximum(np.abs(scipy5), 1e-300)
print(f"Max relative difference from scipy = {np.max(relative_difference5):.3e}")

integration_region5 = 12.0
integrated_pdf5, integration_error5 = integrate.dblquad(
    lambda y, x: float(gaussian_pdf(np.array([x, y]), mu2, Sigma2)),
    -integration_region5,
    integration_region5,
    lambda _: -integration_region5,
    lambda _: integration_region5,
)
print(
    f"Integral over [-{integration_region5:.0f}, {integration_region5:.0f}]^2 = "
    f"{integrated_pdf5:.12f}; quadrature error = {integration_error5:.3e}"
)
print("The square reaches six marginal standard deviations because each marginal variance is 4.")

gauss_samples5 = rng_step5.multivariate_normal(mu2, Sigma2, size=10**4)
sample_correlation5 = np.corrcoef(gauss_samples5, rowvar=False)
contour_grid5 = np.linspace(-12.0, 12.0, 301)
X5, Y5 = np.meshgrid(contour_grid5, contour_grid5, indexing="xy")
positions5 = np.stack([X5, Y5], axis=-1)
density_grid5 = gaussian_pdf(positions5, mu2, Sigma2)
determinant2 = np.linalg.det(Sigma2)
normalizer2 = 1.0 / (2.0 * np.pi * np.sqrt(determinant2))
probabilities5 = np.array([0.50, 0.90, 0.99])
mahalanobis_levels5 = -2.0 * np.log(1.0 - probabilities5)
density_levels5 = normalizer2 * np.exp(-0.5 * mahalanobis_levels5)
contour_order5 = np.argsort(density_levels5)
contour_levels_sorted5 = density_levels5[contour_order5]
probability_labels5 = probabilities5[contour_order5]
mahalanobis_samples5 = mahalanobis_squared(gauss_samples5, mu2, Sigma2)
enclosed_fractions5 = [
    np.mean(mahalanobis_samples5 <= level) for level in mahalanobis_levels5
]
eigenvalues2, eigenvectors2 = np.linalg.eigh(Sigma2)
axis_lengths2 = np.sqrt(eigenvalues2)
rho2 = Sigma2[0, 1] / np.sqrt(Sigma2[0, 0] * Sigma2[1, 1])
print(f"rho = {rho2:.6f}; det(Sigma) = {determinant2:.6f}")
print(f"Sample correlation matrix =\n{sample_correlation5}")
print(f"Mahalanobis-radius-squared levels = {mahalanobis_levels5}")
for probability, fraction in zip(probabilities5, enclosed_fractions5):
    print(
        f"Empirical fraction inside {probability:.0%} contour = {fraction:.6f}; "
        f"difference = {fraction - probability:+.3e}"
    )
print(f"One-standard-deviation ellipse semi-axis lengths = {axis_lengths2}")
print(f"Corresponding eigenvector directions (columns) =\n{eigenvectors2}")

figure5, axis5 = plt.subplots(figsize=(6.2, 5.4))
axis5.scatter(
    gauss_samples5[:, 0],
    gauss_samples5[:, 1],
    s=4,
    alpha=0.18,
    label="10^4 samples",
)
contours5 = axis5.contour(
    X5,
    Y5,
    density_grid5,
    levels=contour_levels_sorted5,
    colors=["tab:blue", "tab:orange", "tab:red"],
    linewidths=1.5,
)
contour_label_map5 = {
    level: f"{probability:.0%}"
    for level, probability in zip(contour_levels_sorted5, probability_labels5)
}
axis5.clabel(contours5, inline=True, fmt=contour_label_map5, fontsize=9)
axis5.set_aspect("equal", adjustable="box")
axis5.set_title("Figure 5: Gaussian scatter and probability contours")
axis5.set_xlabel("x_1")
axis5.set_ylabel("x_2")
axis5.legend()
plt.show()

Figure 5 interpretation. The constant-density curves are ellipses
whose axes are the eigenvector directions of the covariance matrix.
The probability labels are assigned from the two-dimensional
chi-square/Mahalanobis relation, and the empirical counts should agree
with those probabilities within Monte Carlo error.

In [ ]:
rho_values5 = [0.0, 0.5, 0.9, 0.99]
figure6, axes6 = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)
for axis6, rho_value in zip(axes6.ravel(), rho_values5):
    Sigma_rho5 = np.array(
        [[4.0, 4.0 * rho_value], [4.0 * rho_value, 4.0]]
    )
    samples_rho5 = rng_step5.multivariate_normal(mu2, Sigma_rho5, size=5_000)
    density_rho5 = gaussian_pdf(positions5, mu2, Sigma_rho5)
    determinant_rho5 = np.linalg.det(Sigma_rho5)
    rho_axis_level5 = 1.0 / (2.0 * np.pi * np.sqrt(determinant_rho5)) * np.exp(-0.5)
    axis6.scatter(
        samples_rho5[:, 0],
        samples_rho5[:, 1],
        s=3,
        alpha=0.16,
        label="samples",
    )
    axis6.contour(
        X5,
        Y5,
        density_rho5,
        levels=[rho_axis_level5],
        colors=["black"],
        linewidths=1.5,
    )
    axis6.set_aspect("equal", adjustable="box")
    axis6.set_title(f"rho = {rho_value:.2f}; det = {determinant_rho5:.4f}")
    axis6.set_xlabel("x_1")
    axis6.set_ylabel("x_2")
    axis6.legend()
    print(f"rho = {rho_value:.2f}: det(Sigma) = {determinant_rho5:.9f}")
figure6.suptitle("Figure 6: correlation sweep with fixed marginal variances")
plt.show()

Figure 6 interpretation. Increasing the correlation rotates and
narrows the ellipse toward the line where the two measurements move
together. The determinant shrinks because the second measurement
contributes less independent volume when the noises become nearly
redundant.

## Step 6 — Eigendecomposition, decorrelation, and whitening

For a symmetric covariance matrix,

$$\Sigma=V\Lambda V^T,$$

where V is orthogonal and Lambda contains the real eigenvalues.
Rotating by V^T gives a diagonal covariance. Whitening then uses

$$W=\Lambda^{-1/2}V^T,\qquad
W\Sigma W^T=I.$$

If Q is any orthogonal matrix, then QW is also a whitening matrix:

$$(QW)\Sigma(QW)^T=Q I Q^T=I.$$

In [ ]:
print("STEP 6: eigendecomposition and whitening")
eigenvalues2_step6, eigenvectors2_step6 = np.linalg.eigh(Sigma2)
W2_step6 = whitening_matrix(Sigma2)
decorrelated5 = gauss_samples5 @ eigenvectors2_step6
whitened5 = gauss_samples5 @ W2_step6.T
_, covariance_original6 = sample_mean_cov(gauss_samples5)
_, covariance_decorrelated6 = sample_mean_cov(decorrelated5)
_, covariance_whitened6 = sample_mean_cov(whitened5)
print(f"Eigenvalues = {eigenvalues2_step6}")
print(f"Eigenvectors (columns) =\n{eigenvectors2_step6}")
print(f"Original sample covariance =\n{covariance_original6}")
print(f"Decorrelated sample covariance =\n{covariance_decorrelated6}")
print(f"Whitened sample covariance =\n{covariance_whitened6}")
print(f"Original sample correlation =\n{np.corrcoef(gauss_samples5, rowvar=False)}")
print(f"Decorrelated sample correlation =\n{np.corrcoef(decorrelated5, rowvar=False)}")
print(f"Whitened sample correlation =\n{np.corrcoef(whitened5, rowvar=False)}")
print(
    f"||Cov(decorrelated) - Lambda||_F = "
    f"{np.linalg.norm(covariance_decorrelated6 - np.diag(eigenvalues2_step6)):.3e}"
)
whitening_error6 = np.linalg.norm(covariance_whitened6 - np.eye(2), ord="fro")
print(f"||Cov(whitened) - I||_F = {whitening_error6:.3e}")

theta6 = 0.37
Q6 = np.array(
    [[np.cos(theta6), -np.sin(theta6)], [np.sin(theta6), np.cos(theta6)]]
)
rotated_whitener6 = Q6 @ W2_step6
rotated_whitening_error6 = np.linalg.norm(
    rotated_whitener6 @ Sigma2 @ rotated_whitener6.T - np.eye(2),
    ord="fro",
)
print(f"||Q W Sigma W^T Q^T - I||_F = {rotated_whitening_error6:.3e}")

samples4_step6 = rng_step6.multivariate_normal(np.zeros(4), Sigma4, size=10_000)
W4_step6 = whitening_matrix(Sigma4)
whitened4_step6 = samples4_step6 @ W4_step6.T
_, covariance_whitened4 = sample_mean_cov(whitened4_step6)
print(
    f"4D whitening ||Cov(WX)-I||_F = "
    f"{np.linalg.norm(covariance_whitened4 - np.eye(4), ord='fro'):.3e}"
)

figure7, axes7 = plt.subplots(1, 3, figsize=(15, 4.3), constrained_layout=True)
axes7[0].scatter(
    gauss_samples5[:, 0],
    gauss_samples5[:, 1],
    s=3,
    alpha=0.18,
    label="original samples",
)
for index6 in range(2):
    direction6 = eigenvectors2_step6[:, index6]
    length6 = np.sqrt(eigenvalues2_step6[index6])
    axes7[0].plot(
        [-length6 * direction6[0], length6 * direction6[0]],
        [-length6 * direction6[1], length6 * direction6[1]],
        linewidth=2,
        label=f"eigen-axis {index6 + 1}",
    )
axes7[0].set_title("Figure 7a: original")
axes7[1].scatter(
    decorrelated5[:, 0],
    decorrelated5[:, 1],
    s=3,
    alpha=0.18,
    label="decorrelated samples",
)
axes7[1].set_title("Figure 7b: decorrelated")
axes7[2].scatter(
    whitened5[:, 0],
    whitened5[:, 1],
    s=3,
    alpha=0.18,
    label="whitened samples",
)
axes7[2].set_title("Figure 7c: whitened")
for axis7 in axes7:
    axis7.set_aspect("equal", adjustable="box")
    axis7.set_xlabel("coordinate 1")
    axis7.set_ylabel("coordinate 2")
    axis7.legend()
plt.show()

Figure 7 interpretation. The eigenvector lines align with the
original ellipse axes. The rotation removes cross-covariance but leaves
different variances, while whitening additionally scales each
eigendirection so the cloud has identity covariance. The alternative
orthogonal rotation changes orientation but not the whitened covariance.

## Step 7 — The conditional Gaussian

For

$$\begin{bmatrix}X\\Y\end{bmatrix}
\sim\mathcal N\left(
\begin{bmatrix}0\\0\end{bmatrix},
\begin{bmatrix}4&2\\2&4\end{bmatrix}\right),$$

the partitioned-Gaussian formula is

$$Y\mid X=x\sim\mathcal N\left(
\Sigma_{YX}\Sigma_{XX}^{-1}x,\,
\Sigma_{YY}-\Sigma_{YX}\Sigma_{XX}^{-1}\Sigma_{XY}
\right).$$

Here,

$$E[Y\mid X=x]=2\cdot4^{-1}x=\frac12x,$$
$$\operatorname{Var}(Y\mid X)
=4-2\cdot4^{-1}\cdot2=3.$$

The conditional mean is linear and the conditional variance is
constant, which is the structural fact used by the Gaussian linear
estimator.

In [ ]:
print("STEP 7: conditional Gaussian")
joint_samples7 = rng_step7.multivariate_normal(mu2, Sigma2, size=10**6)
joint_sample_correlation7 = np.corrcoef(joint_samples7, rowvar=False)[0, 1]
theoretical_joint_correlation7 = Sigma2[0, 1] / np.sqrt(Sigma2[0, 0] * Sigma2[1, 1])
bins7 = np.linspace(-6.0, 6.0, 49)
centers7, counts7, means7, variances7, means_se7 = binned_moments(
    joint_samples7[:, 0], joint_samples7[:, 1], bins7
)
valid7 = np.isfinite(means7) & np.isfinite(variances7)
slope_fit7 = stats.linregress(centers7[valid7], means7[valid7])
theoretical_conditional_variance7 = 3.0
print(f"Joint sample correlation = {joint_sample_correlation7:.9f}; theory = {theoretical_joint_correlation7:.9f}")
print(
    f"Binned conditional-mean slope = {slope_fit7.slope:.9f} +/- "
    f"{slope_fit7.stderr:.3e}; theory = 0.5"
)
print(
    f"Mean conditional-variance estimate = {np.nanmean(variances7):.9f}; "
    f"theory = {theoretical_conditional_variance7:.9f}"
)
print(
    f"Max absolute variance deviation from theory = "
    f"{np.nanmax(np.abs(variances7 - theoretical_conditional_variance7)):.3e}"
)

figure8, axes8 = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
axes8[0].errorbar(
    centers7[valid7],
    means7[valid7],
    yerr=means_se7[valid7],
    fmt="o",
    markersize=3,
    capsize=2,
    label="binned conditional means",
)
x_line7 = np.linspace(-6.0, 6.0, 200)
axes8[0].plot(x_line7, 0.5 * x_line7, color="black", label="analytic mean: 0.5x")
axes8[0].set_title("Figure 8a: conditional mean")
axes8[0].set_xlabel("x")
axes8[0].set_ylabel("E[Y | X = x]")
axes8[0].legend()
axes8[1].errorbar(
    centers7[valid7],
    variances7[valid7],
    yerr=np.sqrt(2.0 * variances7[valid7] ** 2 / np.maximum(counts7[valid7] - 1, 1)),
    fmt="o",
    markersize=3,
    capsize=2,
    label="binned conditional variances",
)
axes8[1].axhline(3.0, color="black", label="analytic variance: 3")
axes8[1].set_title("Figure 8b: conditional variance")
axes8[1].set_xlabel("x-bin center")
axes8[1].set_ylabel("Var(Y | X = x)")
axes8[1].legend()
plt.show()

Figure 8 interpretation. The binned conditional means follow the
analytic line, and their fitted slope is reported with a standard
error. The conditional-variance bins fluctuate around one horizontal
value rather than changing with x, which is a special property of a
jointly Gaussian pair.

## Step 8 — Fusing two correlated sensors

With X = [X_1, X_2]^T = S 1 + N, Var(S)=sigma_s^2, and N independent
of S,

$$\Sigma_{XX}=\sigma_s^2\mathbf 1\mathbf 1^T+\Sigma_N,\qquad
\Sigma_{SX}=\sigma_s^2\mathbf 1^T.$$

The scalar LMMSE estimate is

$$\widehat S=w^TX,\qquad
w=\Sigma_{XX}^{-1}\Sigma_{XS},$$

with

$$\operatorname{MMSE}
=\sigma_s^2-\Sigma_{SX}\Sigma_{XX}^{-1}\Sigma_{XS}.$$

For the numerical setup in the handout, the determinant of
Sigma_XX is 14 - 8c - c^2, and algebra gives

$$w_2(c)=\frac{4(1-c)}{14-8c-c^2},$$
$$\operatorname{MMSE}(c)
=\frac{4(2-c^2)}{14-8c-c^2}.$$

Thus c = 1 is the weight-crossing point. Differentiating gives

$$\operatorname{MMSE}'(c)
=\frac{32(c-1)(c-2)}{(14-8c-c^2)^2},$$

so c = 1 is a local maximum on the admissible interior, not a
minimum.

In [ ]:
print("STEP 8: fusing two correlated sensors")
sigma_s_squared8 = 4.0
c_values8 = np.array([-1.4, -1.0, -0.5, 0.0, 0.5, 1.0, 1.2, 1.4])

def sensor_metrics8(c_value):
    Sigma_N = np.array([[1.0, c_value], [c_value, 2.0]])
    Sigma_xx = sigma_s_squared8 * np.ones((2, 2)) + Sigma_N
    Sigma_sx = sigma_s_squared8 * np.ones(2)
    weights = lmmse_weights(Sigma_xx, Sigma_sx)
    mmse = sigma_s_squared8 - Sigma_sx @ weights
    equal_weight_mmse = np.array([0.5, 0.5]) @ Sigma_N @ np.array([0.5, 0.5])
    return Sigma_N, Sigma_xx, weights, mmse, equal_weight_mmse


best_single_sensor_mmse8 = sigma_s_squared8 * 1.0 / (sigma_s_squared8 + 1.0)
print(f"Best single-sensor MMSE = {best_single_sensor_mmse8:.9f}")
print("c        rho_N       rho_X       w1          w2          LMMSE MMSE    equal-weight MMSE")
for c_value in c_values8:
    Sigma_N_table8, Sigma_xx_table8, weights8, mmse8, equal_mmse8 = sensor_metrics8(c_value)
    rho_noise8 = Sigma_N_table8[0, 1] / np.sqrt(Sigma_N_table8[0, 0] * Sigma_N_table8[1, 1])
    rho_observed8 = Sigma_xx_table8[0, 1] / np.sqrt(Sigma_xx_table8[0, 0] * Sigma_xx_table8[1, 1])
    print(
        f"{c_value:+.1f}   {rho_noise8: .6f}   {rho_observed8: .6f}   "
        f"{weights8[0]: .9f}  {weights8[1]: .9f}  "
        f"{mmse8:.9f}      {equal_mmse8:.9f}"
    )

sqrt2 = np.sqrt(2.0)
c_plot8 = np.linspace(-sqrt2 + 1e-5, sqrt2 - 1e-5, 600)
weights_plot8 = np.array([sensor_metrics8(c_value)[2] for c_value in c_plot8])
mmse_plot8 = np.array([sensor_metrics8(c_value)[3] for c_value in c_plot8])
equal_mmse_plot8 = np.array([sensor_metrics8(c_value)[4] for c_value in c_plot8])
w2_closed_at_1 = 4.0 * (1.0 - 1.0) / (14.0 - 8.0 - 1.0)
mmse_closed_at_1 = 4.0 * (2.0 - 1.0) / (14.0 - 8.0 - 1.0)
mmse_second_derivative_at_1 = -32.0 / 25.0
variance_difference_14 = 3.0 - 2.0 * 1.4
print(f"Admissible PSD range: |c| <= sqrt(2) = {sqrt2:.9f}")
print(f"Strictly positive-definite interior used by inverse formulas: |c| < {sqrt2:.9f}")
print(f"w2(1) = {w2_closed_at_1:.9f}; MMSE(1) = {mmse_closed_at_1:.9f}")
print(
    f"MMSE'(1) = 0 by the closed form; MMSE''(1) = "
    f"{mmse_second_derivative_at_1:.9f}, so c=1 is a local maximum."
)
print(f"Var(X1 - X2) at c=1.4 = {variance_difference_14:.9f}")
mmse_c0 = sensor_metrics8(0.0)[3]
mmse_c14 = sensor_metrics8(1.4)[3]
print(f"MMSE(c=0) = {mmse_c0:.9f}; MMSE(c=1.4) = {mmse_c14:.9f}")
print(
    f"MMSE at both PSD endpoints approaches 0; c=-1 gives "
    f"{sensor_metrics8(-1.0)[3]:.9f}"
)

figure9, axes9 = plt.subplots(1, 2, figsize=(13, 4.4), constrained_layout=True)
axes9[0].plot(c_plot8, weights_plot8[:, 0], label="w1")
axes9[0].plot(c_plot8, weights_plot8[:, 1], label="w2")
axes9[0].axvline(1.0, color="black", linestyle="--", label="sign change at c=1")
axes9[0].axhline(0.0, color="gray", linewidth=0.8)
axes9[0].set_title("Figure 9a: LMMSE weights")
axes9[0].set_xlabel("c")
axes9[0].set_ylabel("weight")
axes9[0].legend()
axes9[1].plot(c_plot8, mmse_plot8, label="LMMSE MMSE")
axes9[1].plot(c_plot8, np.full_like(c_plot8, best_single_sensor_mmse8), "--", label="best single sensor")
axes9[1].plot(c_plot8, equal_mmse_plot8, label="naive equal weighting")
axes9[1].axvline(1.0, color="black", linestyle="--", label="c=1")
axes9[1].set_title("Figure 9b: estimation error")
axes9[1].set_xlabel("c")
axes9[1].set_ylabel("MMSE")
axes9[1].legend()
plt.show()

n_mc8 = 10**6
for c_value in [0.8, 1.4]:
    Sigma_N8, _, weights_mc8, analytic_mmse8, _ = sensor_metrics8(c_value)
    s_trials8 = rng_step8.normal(0.0, np.sqrt(sigma_s_squared8), size=n_mc8)
    noise_trials8 = rng_step8.multivariate_normal(np.zeros(2), Sigma_N8, size=n_mc8)
    x_trials8 = s_trials8[:, None] + noise_trials8
    estimates8 = x_trials8 @ weights_mc8
    squared_errors8 = (estimates8 - s_trials8) ** 2
    simulated_mmse8 = np.mean(squared_errors8)
    simulated_mmse_se8 = np.std(squared_errors8, ddof=1) / np.sqrt(n_mc8)
    print(
        f"Monte Carlo c={c_value:.1f}: simulated MSE = {simulated_mmse8:.9f}, "
        f"SE = {simulated_mmse_se8:.3e}, analytic MMSE = {analytic_mmse8:.9f}, "
        f"difference = {simulated_mmse8 - analytic_mmse8:+.3e}"
    )

print(
    "Systems-engineer conclusion: highly correlated noise is useful when its "
    "correlation is accurately known, because the estimator can reject the "
    "common component. Independent noise is safer when the correlation model "
    "is uncertain; specify the covariance, not just each sensor variance."
)

Figure 9 interpretation. After the correlation crosses the point where
the second sensor’s noise aligns with the first sensor’s noise, the
optimal estimator subtracts the second measurement. That negative
weight cancels common noise; it is not a coding error. The difference
signal becomes especially informative near a singular noise covariance,
while the equal-weight baseline cannot exploit that cancellation.

## Bonus A — Generating correlated vectors from scratch

If z has identity covariance and L is a Cholesky factor,

$$\operatorname{Cov}(Lz)=L\operatorname{Cov}(z)L^T=LL^T=\Sigma.$$

An eigendecomposition gives the alternative transform
V Lambda^(1/2) z. Cholesky is usually cheaper for a positive definite
matrix; the eigendecomposition is more informative for geometry and
can handle positive-semidefinite matrices with zero eigenvalues.

In [ ]:
print("BONUS A: correlated samples from Cholesky and eigen factors")
for covariance_name, covariance_bonus in [("2D", Sigma2), ("4D", Sigma4)]:
    chol_draws = cholesky_samples(BONUS_RNG_A, covariance_bonus, 100_000)
    eig_draws = eigen_samples(BONUS_RNG_A, covariance_bonus, 100_000)
    _, chol_covariance = sample_mean_cov(chol_draws)
    _, eig_covariance = sample_mean_cov(eig_draws)
    chol_correlation = np.corrcoef(chol_draws, rowvar=False)
    eig_correlation = np.corrcoef(eig_draws, rowvar=False)
    print(
        f"{covariance_name}: Cholesky covariance error = "
        f"{np.linalg.norm(chol_covariance - covariance_bonus, ord='fro'):.3e}; "
        f"eigen covariance error = "
        f"{np.linalg.norm(eig_covariance - covariance_bonus, ord='fro'):.3e}"
    )
    print(f"{covariance_name}: Cholesky sample correlation =\n{chol_correlation}")
    print(f"{covariance_name}: eigen-factor sample correlation =\n{eig_correlation}")

singular_bonus = np.array([[1.0, 1.0], [1.0, 1.0]])
try:
    np.linalg.cholesky(singular_bonus)
except np.linalg.LinAlgError as error:
    print(f"Cholesky on singular covariance fails as expected: {error}")

Bonus A interpretation. Both factorizations reproduce the target
covariance up to finite-sample error. Cholesky rejects the singular
example because it requires a strictly positive-definite matrix.

## Bonus B — Principal components on real data

The built-in scikit-learn iris dataset supplies four numeric
measurements. The code compares PCA on the raw columns with PCA after
standardizing each column. The standardized version gives each
measurement equal variance weight; the raw version preserves the
original physical units.

In [ ]:
print("BONUS B: PCA on the scikit-learn iris dataset")
from sklearn.datasets import load_iris

iris = load_iris()
iris_data = iris.data.astype(float)
iris_centered = iris_data - np.mean(iris_data, axis=0)
iris_covariance = np.cov(iris_centered, rowvar=False, ddof=1)
iris_values, iris_vectors = np.linalg.eigh(iris_covariance)
iris_order = np.argsort(iris_values)[::-1]
iris_values = iris_values[iris_order]
iris_vectors = iris_vectors[:, iris_order]
raw_fraction_first_two = np.sum(iris_values[:2]) / np.sum(iris_values)
raw_projection = iris_centered @ iris_vectors[:, :2]

iris_standardized = iris_centered / np.std(iris_data, axis=0, ddof=1)
standardized_covariance = np.cov(iris_standardized, rowvar=False, ddof=1)
raw_correlation_iris = iris_covariance / np.outer(np.sqrt(np.diag(iris_covariance)), np.sqrt(np.diag(iris_covariance)))
standardized_correlation_iris = standardized_covariance / np.outer(np.sqrt(np.diag(standardized_covariance)), np.sqrt(np.diag(standardized_covariance)))
standardized_values, standardized_vectors = np.linalg.eigh(standardized_covariance)
standardized_order = np.argsort(standardized_values)[::-1]
standardized_values = standardized_values[standardized_order]
standardized_vectors = standardized_vectors[:, standardized_order]
standardized_fraction_first_two = np.sum(standardized_values[:2]) / np.sum(standardized_values)
standardized_projection = iris_standardized @ standardized_vectors[:, :2]

print(f"Raw PCA fraction in first two components = {raw_fraction_first_two:.9f}")
print(f"Raw-data correlation matrix =\n{raw_correlation_iris}")
print(f"Standardized-data correlation matrix =\n{standardized_correlation_iris}")
print(
    f"Standardized PCA fraction in first two components = "
    f"{standardized_fraction_first_two:.9f}"
)

figure_bonus_b, axes_bonus_b = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for class_value, label in enumerate(iris.target_names):
    mask = iris.target == class_value
    axes_bonus_b[0].scatter(
        raw_projection[mask, 0],
        raw_projection[mask, 1],
        s=18,
        label=label,
    )
    axes_bonus_b[1].scatter(
        standardized_projection[mask, 0],
        standardized_projection[mask, 1],
        s=18,
        label=label,
    )
axes_bonus_b[0].set_title("Bonus Figure B1a: raw PCA")
axes_bonus_b[1].set_title("Bonus Figure B1b: standardized PCA")
for axis_bonus_b in axes_bonus_b:
    axis_bonus_b.set_xlabel("PC 1")
    axis_bonus_b.set_ylabel("PC 2")
    axis_bonus_b.legend()
plt.show()

Bonus B interpretation. Standardization changes the covariance
geometry because it removes the original unit and scale differences. I
would trust raw PCA when the measurement units and variances have
physical meaning; I would trust standardized PCA when the columns
should contribute on an equal scale.

## Final clean-kernel checklist

The final output below records the runtime and key automated checks.
Before submission, also restart the kernel and run all cells, add team
contributions to the README, and replace the placeholder seed if
needed.

In [ ]:
runtime_seconds = time.perf_counter() - START_TIME
print(f"Total runtime = {runtime_seconds:.3f} seconds")
print(f"Runtime under five minutes = {runtime_seconds < 300.0}")
print(f"Required function smoke checks = {all([
    is_valid_covariance(Sigma4),
    np.allclose(gaussian_pdf(np.zeros(2), mu2, Sigma2), 1.0 / (2.0 * np.pi * np.sqrt(12.0))),
    np.allclose(whitening_matrix(Sigma2) @ Sigma2 @ whitening_matrix(Sigma2).T, np.eye(2)),
    np.allclose(lmmse_weights(np.eye(2), np.array([1.0, 2.0])), np.array([1.0, 2.0])),
])}")
print("Notebook run reached the end without an exception.")